### Block 0: install required packages

In [6]:
!pip install -q requests datasets scikit-learn tqdm pandas


### Block 1: imports and API configuration

In [7]:

import os
import getpass
import requests

from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Read your DeepSeek API key (from DeepSeek dashboard)
DEEPSEEK_API_KEY = getpass.getpass("Paste your DeepSeek API key here: ")
os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY

# Read your Hugging Face token (for gated dataset access)
HF_TOKEN = getpass.getpass("Paste your Hugging Face token here: ")
os.environ["HF_TOKEN"] = HF_TOKEN

# DeepSeek chat completion endpoint and model name
DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"
MODEL_NAME = "deepseek-chat"  # DeepSeek V3 chat model


Paste your DeepSeek API key here: ··········
Paste your Hugging Face token here: ··········


### Block 2: load TheFinAI/flare-fiqasa dataset

In [8]:
# Load the FLARE-FIQASA dataset (requires that you accepted the terms on the HF website)
raw_ds = load_dataset("TheFinAI/flare-fiqasa", token=os.environ["HF_TOKEN"])

# Prefer the "test" split if available
if "test" in raw_ds:
    split_name = "test"
elif "validation" in raw_ds:
    split_name = "validation"
else:
    split_name = "train"

dataset = raw_ds[split_name]

print(dataset)
print("Using split:", split_name)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])

# Detect text and label columns
TEXT_FIELD = None
LABEL_FIELD = None

text_candidates = ["text", "input", "sentence", "post", "query"]
label_candidates = ["answer", "label", "target", "output", "gold"]

for name in text_candidates:
    if name in dataset.column_names:
        TEXT_FIELD = name
        break

for name in label_candidates:
    if name in dataset.column_names:
        LABEL_FIELD = name
        break

print("Detected text field:", TEXT_FIELD)
print("Detected label field:", LABEL_FIELD)

assert TEXT_FIELD is not None, "Could not find a text column."
assert LABEL_FIELD is not None, "Could not find a label column."


data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Using split: test
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}
Detected text field: text
Detected label field: answer


### Block 3: label space and normalization helper

In [9]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map raw model output to one of: 'negative', 'neutral', 'positive'.
    """
    text = (raw or "").strip().lower()

    # Exact match first
    for label in LABELS:
        if label in text:
            return label

    # Short forms
    if "pos" in text:
        return "positive"
    if "neg" in text:
        return "negative"

    # Fallback
    return "neutral"


### Block 4: DeepSeek HTTP helpers

In [10]:
def build_prompt(text: str) -> str:
    """
    Build the instruction prompt for sentiment classification.
    """
    return (
        "You are an expert financial sentiment classifier.\n\n"
        "Classify the sentiment of the following financial post as exactly one of:\n"
        "negative, neutral, positive.\n\n"
        "Return only one word: negative, neutral, or positive.\n\n"
        f"Text: {text}\n"
        "Answer:"
    )


def query_deepseek(prompt: str) -> str:
    """
    Send a chat completion request to the DeepSeek API and return the raw text response.
    """
    headers = {
        "Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a financial sentiment classifier. "
                    "Always reply with exactly one word: negative, neutral, or positive."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        "max_tokens": 8,
        "temperature": 0.0,
    }

    response = requests.post(
        DEEPSEEK_API_URL,
        headers=headers,
        json=payload,
        timeout=60,
    )

    response.raise_for_status()  # raise error if status is not 2xx

    data = response.json()
    return data["choices"][0]["message"]["content"]


def predict_label(text: str) -> str:
    """
    High-level helper: build prompt, call DeepSeek, normalize label.
    """
    prompt = build_prompt(text)
    raw = query_deepseek(prompt)
    return normalize_prediction(raw)


### Block 5: run evaluation loop over FLARE-FIQASA

In [11]:
y_true = []
y_pred = []

for example in tqdm(dataset, total=len(dataset)):
    text = example[TEXT_FIELD]
    true_label = str(example[LABEL_FIELD]).strip().lower()

    y_true.append(true_label)

    pred_label = predict_label(text)
    y_pred.append(pred_label)


100%|██████████| 235/235 [04:43<00:00,  1.21s/it]


### Block 6: compute metrics and inspect predictions

In [12]:
# Overall accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

# Build DataFrame for inspection
df = pd.DataFrame({
    "text": [ex[TEXT_FIELD] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))


Accuracy: 0.7362

Classification report:
              precision    recall  f1-score   support

    negative     0.9048    0.7500    0.8201        76
     neutral     0.2206    0.8333    0.3488        18
    positive     0.9712    0.7163    0.8245       141

    accuracy                         0.7362       235
   macro avg     0.6988    0.7665    0.6645       235
weighted avg     0.8922    0.7362    0.7867       235


First 10 predictions:
                                                                                                                            text     gold     pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative negative
                                                                         Legal & General share price: Finance chief to step down negative  neutral
                                                                 Kingfisher share price slides on cost to implemen